# EXP01: Keltner max_hold 全套实验

**合并**: NB01 + NB04 + NB05

1. KC max_hold 扫描 (11 变体)
2. 出场原因深度对比 (baseline vs champion)
3. 冠军 K-Fold 验证

**动机**: Timeout 60 bars = 769笔 -$12,287，是最大单项亏损源。赢家中位 6 bars，输家中位 46 bars。

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked
import config
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Part 1: KC max_hold 扫描

In [ ]:
original_mh = config.STRATEGIES['keltner']['max_hold_bars']
results = []

for mh in [4, 6, 8, 10, 12, 16, 20, 24, 32, 40, 60]:
    config.STRATEGIES['keltner']['max_hold_bars'] = mh
    r = run_variant(data, f"KC_mh={mh}", **LIVE_KWARGS)
    r['kc_max_hold'] = mh
    results.append(r)

config.STRATEGIES['keltner']['max_hold_bars'] = original_mh
print_ranked(results)

In [ ]:
for r in results:
    trades = r['_trades']
    df = pd.DataFrame([{'exit_reason': t.exit_reason, 'pnl': t.pnl, 'strategy': t.strategy} for t in trades])
    print(f"\n=== KC max_hold={r['kc_max_hold']} ===")
    print(df.groupby('exit_reason')['pnl'].agg(['sum', 'mean', 'count']))

## Part 2: 出场原因深度对比 (Baseline vs Champion)

In [ ]:
best = sorted(results, key=lambda r: r['sharpe'], reverse=True)[0]
BEST_KC_MH = best['kc_max_hold']
print(f"Champion: KC max_hold={BEST_KC_MH}, Sharpe={best['sharpe']:.2f}, PnL=${best['total_pnl']:.0f}")

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)

def analyze_exits(trades, label):
    df = pd.DataFrame([{
        'exit_reason': t.exit_reason.split(':')[0] if ':' in t.exit_reason else t.exit_reason,
        'pnl': t.pnl, 'strategy': t.strategy, 'direction': t.direction, 'bars_held': t.bars_held,
    } for t in trades])
    print(f"\n{'='*60}")
    print(f"  {label} ({len(trades)} trades, PnL=${sum(t.pnl for t in trades):.0f})")
    print(f"{'='*60}")
    summary = df.groupby('exit_reason').agg(
        count=('pnl', 'count'), total_pnl=('pnl', 'sum'), avg_pnl=('pnl', 'mean'),
        win_rate=('pnl', lambda x: (x > 0).mean()), avg_bars=('bars_held', 'mean'),
    ).round(2)
    print(summary)
    print("\n--- By Strategy ---")
    print(df.groupby(['strategy', 'exit_reason'])['pnl'].agg(['sum', 'count']).round(2))

analyze_exits(baseline['_trades'], "Baseline")
analyze_exits(best['_trades'], f"Champion KC_mh={BEST_KC_MH}")

In [ ]:
for label, r in [('Baseline', baseline), ('Champion', best)]:
    bars = [t.bars_held for t in r['_trades']]
    pnls = [t.pnl for t in r['_trades']]
    print(f"{label}: median_bars={np.median(bars):.0f}, mean_bars={np.mean(bars):.1f}, "
          f"winners_median={np.median([b for b,p in zip(bars,pnls) if p>0]):.0f}, "
          f"losers_median={np.median([b for b,p in zip(bars,pnls) if p<0]):.0f}")

## Part 3: 冠军 K-Fold 验证

In [ ]:
print("=== Baseline K-Fold ===")
base_folds = run_kfold(data, LIVE_KWARGS, label_prefix="Base_")

print(f"\n=== Champion KC_mh={BEST_KC_MH} K-Fold ===")
config.STRATEGIES['keltner']['max_hold_bars'] = BEST_KC_MH
champ_folds = run_kfold(data, LIVE_KWARGS, label_prefix="Champ_")
config.STRATEGIES['keltner']['max_hold_bars'] = original_mh

print("\n=== Comparison ===")
for b, c in zip(base_folds, champ_folds):
    print(f"{b['fold']}: Base={b['sharpe']:.2f}  Champ={c['sharpe']:.2f}  D={c['sharpe']-b['sharpe']:+.2f}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp01_results.json', 'w') as f:
    json.dump(sanitize_for_json(results), f, indent=2)
print('Saved to data/exp01_results.json')